# E-Commerce Analytics 360
## Étape 3 — SQL Analytics, KPIs & Segmentation RFM

> **Prérequis** : avoir complété l'étape 2. Le dataset `fact_ecommerce_analytics` doit être disponible.

| | |
|---|---|
| **Niveau** | Avancé |
| **Outils** | Python, DuckDB, Matplotlib |
| **Durée estimée** | 2h à 3h |

---
## 🦆 C'est quoi DuckDB ?

DuckDB est une **base de données SQL embarquée**, conçue pour l'analyse de données.

Pense-y comme **SQLite, mais optimisé pour faire de l'analytique** — là où SQLite est pensé pour les applications transactionnelles.

### Pourquoi DuckDB dans ce notebook ?

Dans l'étape précédente, on utilisait SQL Server — une base installée localement sur ton PC. Dans Colab, il est impossible de se connecter à `localhost`. DuckDB résout ce problème élégamment :

| | SQL Server | DuckDB |
|---|---|---|
| Installation | Lourd (500MB+) | `pip install duckdb` |
| Fonctionnement | Serveur séparé | Directement dans Python |
| Compatibilité Colab | ❌ | ✅ |
| Syntaxe SQL | SQL Server dialect | Standard SQL (proche de PostgreSQL) |
| Window functions | ✅ | ✅ |
| CTEs | ✅ | ✅ |
| Lire un DataFrame pandas | ❌ (besoin d'export) | ✅ Directement |

### La killer feature : requêter des DataFrames pandas directement

```python
import duckdb

# df est un DataFrame pandas chargé normalement
result = duckdb.sql("SELECT canal, SUM(revenue) FROM df GROUP BY canal").df()
```

Pas d'export, pas de serveur — DuckDB lit le DataFrame pandas en mémoire et renvoie le résultat comme un nouveau DataFrame (`.df()`).

### Différences syntaxiques avec SQL Server

| Opération | SQL Server | DuckDB |
|---|---|---|
| Extraire le mois | `FORMAT(order_date, 'yyyy-MM')` | `strftime(order_date, '%Y-%m')` |
| Limiter les résultats | `TOP 10` | `LIMIT 10` |
| Arrondir | `ROUND(x, 0)` | `ROUND(x, 0)` |
| Window functions | identique | identique |
| CTEs | identique | identique |
| `LAG()`, `RANK()`, `ROW_NUMBER()` | identique | identique |

---
## 0. Mise en place de l'environnement

In [ ]:
!pip install duckdb --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import duckdb
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.2f}".format)

COLORS = {
    "primary":   "#534AB7",
    "secondary": "#1D9E75",
    "warning":   "#EF9F27",
    "danger":    "#E24B4A",
    "neutral":   "#888780",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F9F9F8",
    "axes.grid":        True,
    "grid.alpha":       0.35,
    "font.size":        11,
})

print("Environnement prêt ✅")

---
## 1. Chargement du dataset analytique

On charge le dataset produit dans l'étape 2 (`fact_ecommerce_analytics`).

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/[username]/DataProjectLab/main/projets/ecommerce_analytics/data/"

# À compléter — charge le dataset nettoyé produit en étape 2
# df = pd.read_csv(BASE_URL + "fact_ecommerce_analytics.csv", parse_dates=["order_date"])

# Vérification
# print(f"Dataset chargé : {len(df):,} lignes")
# print(f"Colonnes : {df.columns.tolist()}")

### Comment utiliser DuckDB dans ce notebook

Pour chaque question SQL, tu utiliseras ce pattern :

```python
result = duckdb.sql("""
    SELECT ...
    FROM df          -- 'df' est ton DataFrame pandas, DuckDB le lit directement
    WHERE ...
""").df()           -- .df() convertit le résultat en DataFrame pandas

result
```

> 💡 Le filtre `WHERE order_status = 'Livree'` est à appliquer sur toutes les questions financières.

---
# PARTIE 1 — SQL Analytics avec DuckDB

## Bloc A — KPIs Fondamentaux

**Objectif** : calculer les indicateurs clés de performance globaux de la plateforme.

### Question A1 — KPIs globaux
Dans **une seule requête**, calcule :
- CA total (`SUM(revenue)`)
- Marge totale (`SUM(margin)`)
- Taux de marge en % (marge / CA × 100)
- Nombre de commandes distinctes

In [ ]:
# À compléter
# duckdb.sql("""
#     SELECT ...
#     FROM df
#     WHERE order_status = 'Livree'
# """).df()

### Question A2 — Panier moyen
Calcule le panier moyen par commande.

💡 *Deux étapes : d'abord agrège le revenue par `order_id`, ensuite fais la moyenne. Utilise une sous-requête ou une CTE.*

In [ ]:
# À compléter

### Question A3 — Clients actifs
Compte le nombre de clients distincts ayant au moins une commande livrée.

In [ ]:
# À compléter

---
## Bloc B — Analyse Temporelle

**Objectif** : comprendre l'évolution du CA dans le temps et détecter les anomalies.

### Question B4 — CA et marge mensuels
Pour chaque mois : CA, marge, taux de marge, nombre de commandes.

💡 *Dans DuckDB : `strftime(order_date, '%Y-%m') AS mois`*

In [ ]:
# À compléter

### Question B5 — Top 3 mois par CA

💡 *Dans DuckDB : `ORDER BY ca DESC LIMIT 3`*

In [ ]:
# À compléter

### Question B6 — Anomalies : CA en hausse mais marge en baisse ⭐

Identifie les mois où le CA augmente par rapport au mois précédent **mais** où la marge baisse.

💡 *Utilise la fonction `LAG()` — une window function.*

**📌 Point pédagogique** :
> `LAG(ca) OVER (ORDER BY mois)` retourne la valeur de la ligne précédente.  
> C'est une **window function** : elle ne groupe pas les lignes, elle les compare entre elles.  
> La fenêtre `OVER (ORDER BY mois)` définit l'ordre de comparaison.

In [ ]:
# Structure guidée
# duckdb.sql("""
#     WITH monthly AS (
#         SELECT
#             strftime(order_date, '%Y-%m') AS mois,
#             SUM(revenue) AS ca,
#             SUM(margin)  AS marge
#         FROM df
#         WHERE order_status = 'Livree'
#         GROUP BY strftime(order_date, '%Y-%m')
#     )
#     SELECT
#         mois, ca, marge,
#         ca    - LAG(ca)    OVER (ORDER BY mois) AS delta_ca,
#         marge - LAG(marge) OVER (ORDER BY mois) AS delta_marge
#     FROM monthly
#     -- TODO : filtre sur delta_ca > 0 AND delta_marge < 0
#     ORDER BY mois
# """).df()

---
## Bloc C — Produits & Catégories

### Question C7 — Performance par catégorie avec ranking ⭐

Pour chaque catégorie : CA total, marge, taux de marge, quantité vendue, **rang CA**, **rang marge**.

**📌 Point pédagogique** :
> `RANK() OVER (ORDER BY SUM(revenue) DESC)` classe sans supprimer de lignes.  
> Tu vois le rang de toutes les catégories en même temps — c'est la différence avec `LIMIT`.

In [ ]:
# À compléter

### Question C8 — Top 10 marques par quantité vendue
Avec leur CA et leur taux de marge.

In [ ]:
# À compléter

---
## Bloc D — Analyse Clients

### Question D9 — Top 20 clients par CA
Nombre de commandes, CA total, marge totale, panier moyen, rang CA, rang marge.

In [ ]:
# À compléter

### Question D10 — Clients avec panier moyen élevé
Identifie les clients avec un panier moyen supérieur à 200€.

💡 *Utilise la clause `HAVING` pour filtrer après le `GROUP BY`.*

In [ ]:
# À compléter

---
## Bloc E — Canaux d'Acquisition

### Question E11 — Performance complète par canal ⭐

CA, marge, taux de marge, nb commandes, panier moyen, **part du CA total en %**.

**📌 Point pédagogique** :
> `SUM(SUM(revenue)) OVER ()` est une window function imbriquée.  
> `OVER()` vide = pas de partition = total sur toutes les lignes du résultat.  
> C'est le moyen de calculer un pourcentage du total dans une seule requête.

In [ ]:
# Structure guidée
# duckdb.sql("""
#     SELECT
#         canal,
#         ROUND(SUM(revenue), 0) AS ca,
#         ...
#         ROUND(SUM(revenue) * 100.0 / SUM(SUM(revenue)) OVER (), 1) AS part_ca_pct
#     FROM df
#     WHERE order_status = 'Livree'
#     GROUP BY canal
#     ORDER BY ca DESC
# """).df()

---
## Bloc F — Risque & Qualité Opérationnelle

### Question F12 — KPIs de risque globaux

Dans une seule requête : taux de commandes à risque, délai moyen, % non livrées, nb annulées, nb remboursées.

💡 *Utilise `CASE WHEN` pour compter conditionnellement.*

In [ ]:
# Structure guidée
# duckdb.sql("""
#     SELECT
#         SUM(CASE WHEN order_status = 'Annulee'    THEN 1 ELSE 0 END) AS nb_annulees,
#         SUM(CASE WHEN order_status = 'Remboursee' THEN 1 ELSE 0 END) AS nb_remboursees,
#         -- TODO : complète les autres KPIs
#     FROM df
# """).df()

### Question F13 — Risque par canal
Quel canal génère le plus de commandes à risque ?

In [ ]:
# À compléter

---
## Bloc G — SQL Avancé ⭐⭐

### Question G14 — Ranking mensuel des catégories ⭐⭐

Pour chaque mois, classe les catégories par CA et affiche le **top 3 par mois**.

**📌 Point pédagogique** :
> `ROW_NUMBER() OVER (PARTITION BY mois ORDER BY ca DESC)` :  
> - `PARTITION BY mois` : recommence la numérotation à 1 pour chaque mois  
> - `ORDER BY ca DESC` : rang 1 = plus fort CA  
> - `ROW_NUMBER` ne produit jamais de doublons (contrairement à `RANK`)

In [ ]:
# Structure guidée
# duckdb.sql("""
#     WITH monthly_cat AS (
#         SELECT
#             strftime(order_date, '%Y-%m') AS mois,
#             categorie,
#             ROUND(SUM(revenue), 0) AS ca
#         FROM df
#         WHERE order_status = 'Livree'
#         GROUP BY strftime(order_date, '%Y-%m'), categorie
#     ),
#     ranked AS (
#         SELECT *,
#             ROW_NUMBER() OVER (PARTITION BY mois ORDER BY ca DESC) AS rang_mensuel
#         FROM monthly_cat
#     )
#     SELECT * FROM ranked
#     WHERE rang_mensuel <= 3
#     ORDER BY mois, rang_mensuel
# """).df()

### Question G15 — Variation mensuelle du CA en % ⭐⭐

CA du mois, CA du mois précédent, variation en %.

💡 *Utilise `LAG()` pour récupérer le mois précédent.*

In [ ]:
# À compléter

### Question G16 — Clients au-dessus de la moyenne globale ⭐⭐

Identifie les clients dont le CA total est supérieur à la moyenne globale des clients.

💡 *Utilise deux CTEs : une pour le CA par client, une pour la moyenne globale, puis un `CROSS JOIN`.*

In [ ]:
# Structure guidée
# duckdb.sql("""
#     WITH client_ca AS (
#         SELECT customer_id, segment_client,
#                ROUND(SUM(revenue), 0) AS ca_client
#         FROM df
#         WHERE order_status = 'Livree'
#         GROUP BY customer_id, segment_client
#     ),
#     moyenne AS (
#         SELECT AVG(ca_client) AS moy FROM client_ca
#     )
#     SELECT c.*, m.moy AS moyenne_globale,
#            ROUND(c.ca_client - m.moy, 0) AS ecart_moyenne
#     FROM client_ca c
#     CROSS JOIN moyenne m
#     WHERE c.ca_client > m.moy
#     ORDER BY c.ca_client DESC
# """).df()

---
# PARTIE 2 — Segmentation RFM en Python

## Pourquoi Python pour la segmentation RFM ?

SQL est excellent pour les agrégations et les jointures. Mais pour des opérations comme :
- Le découpage en quantiles (`pd.qcut`)
- L'application de règles métier complexes (`apply`)
- La visualisation avancée (scatter, donut)

pandas + matplotlib est beaucoup plus adapté.

> Dans un projet réel, on calcule souvent les métriques brutes en SQL et on fait la segmentation et la visualisation en Python. C'est exactement ce qu'on fait ici.

---
## Étape 1 — Préparation du dataset

In [ ]:
# On travaille uniquement sur les commandes livrées
# df_liv = df[df["order_status"] == "Livree"].copy()

# print(f"Dataset chargé    : {len(df):,} lignes")
# print(f"Commandes livrées : {len(df_liv):,} lignes")
# print(f"Période : {df_liv['order_date'].min().date()} → {df_liv['order_date'].max().date()}")

---
## Étape 2 — Calcul des métriques RFM

| Métrique | Définition | Calcul |
|---|---|---|
| **R** — Recency | Depuis combien de jours le client a commandé | `date_ref - max(order_date)` |
| **F** — Frequency | Nombre de commandes distinctes | `COUNT(DISTINCT order_id)` |
| **M** — Monetary | Montant total dépensé | `SUM(revenue)` |

> **Important** : pour la Recency, un petit nombre = client récent = bon signe. Le score R sera donc **inversé** lors du scoring.

In [ ]:
# Date de référence = lendemain du dernier achat dans le dataset
# DATE_REF = df_liv["order_date"].max() + pd.Timedelta(days=1)
# print(f"Date de référence : {DATE_REF.date()}")

# TODO : Calcule les métriques RFM par client
# rfm = (
#     df_liv.groupby("customer_id")
#     .agg(
#         last_order   = ("order_date",     "???"),   # date de la dernière commande
#         frequency    = ("order_id",       "???"),   # nombre de commandes distinctes
#         monetary     = ("revenue",        "???"),   # CA total
#         segment      = ("segment_client", "first"),
#         pays         = ("pays",           "first")
#     )
#     .reset_index()
# )

# TODO : Calcule la recency en nombre de jours
# rfm["recency"] = (DATE_REF - rfm["last_order"]).dt.days
# rfm = rfm.drop(columns="last_order")

# print(f"\nDataset RFM : {len(rfm):,} clients")
# rfm.head(5)

---
## Étape 3 — Calcul des scores R, F, M

On divise chaque métrique en **5 classes égales (quantiles)**. Chaque client reçoit un score de 1 à 5.

| Score | R (Recency) | F (Frequency) | M (Monetary) |
|---|---|---|---|
| 5 | Très récent ✅ | Très fréquent ✅ | Gros dépensier ✅ |
| 1 | Inactif depuis longtemps ❌ | Rarement ❌ | Petit panier ❌ |

> **Pourquoi `rank(method='first')` pour F et M ?**  
> `pd.qcut` échoue s'il y a trop de doublons. `rank(method='first')` transforme les valeurs en rangs uniques pour que `qcut` puisse fonctionner.

In [ ]:
# Score R : inversé (récent = bon = score 5)
# rfm["R_score"] = pd.qcut(
#     rfm["recency"],
#     q=5,
#     labels=[5, 4, 3, 2, 1]  # inversé : petit recency = score élevé
# ).astype(int)

# TODO : Score F — fréquent = bon = score 5
# rfm["F_score"] = pd.qcut(
#     rfm["frequency"].rank(method="first"),
#     q=5,
#     labels=# TODO : [1,2,3,4,5] dans quel ordre ?
# ).astype(int)

# TODO : Score M — même logique que F
# rfm["M_score"] = # TODO

# Score RFM combiné
# rfm["rfm_score"] = rfm["R_score"].astype(str) + rfm["F_score"].astype(str) + rfm["M_score"].astype(str)
# rfm["rfm_total"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

# rfm[["customer_id", "recency", "frequency", "monetary",
#      "R_score", "F_score", "M_score", "rfm_score", "rfm_total"]].head(10)

---
## Étape 4 — Segmentation business

| Segment | Règle | Description |
|---|---|---|
| **Champions** | R≥4 ET F≥4 ET M≥4 | Récents, fréquents, gros dépensiers |
| **Fidèles** | R≥3 ET F≥3 | Commandent régulièrement |
| **Gros dépensiers occasionnels** | M≥4 ET F≤2 | Panier élevé mais peu fréquents |
| **Nouveaux prometteurs** | R≥4 ET F≤2 | Premier(s) achat récent |
| **À réactiver** | R≤2 ET rfm_total≥8 | Inactifs mais bonne valeur historique |
| **Dormants** | Tous les autres | Faible sur les 3 dimensions |

In [ ]:
# TODO : Complète la fonction de segmentation
# def segment_rfm(row):
#     r     = row["R_score"]
#     f     = row["F_score"]
#     m     = row["M_score"]
#     total = row["rfm_total"]
#
#     if r >= 4 and f >= 4 and m >= 4:
#         return "Champions"
#     elif r >= 3 and f >= 3:
#         return "Fidèles"
#     elif m >= 4 and f <= 2:
#         return # TODO
#     elif r <= 2 and total >= 8:
#         return # TODO
#     elif r >= 4 and f <= 2:
#         return # TODO
#     else:
#         return "Dormants"
#
# rfm["segment_rfm"] = rfm.apply(segment_rfm, axis=1)
#
# dist = rfm["segment_rfm"].value_counts().reset_index()
# dist.columns = ["segment", "nb_clients"]
# dist["pct"] = (dist["nb_clients"] / dist["nb_clients"].sum() * 100).round(1)
# display(dist)

---
## Étape 5 — Visualisation

In [ ]:
# colors_seg = {
#     "Champions":                    COLORS["secondary"],
#     "Fidèles":                      COLORS["primary"],
#     "Gros dépensiers occasionnels": COLORS["warning"],
#     "Nouveaux prometteurs":         "#5DCAA5",
#     "À réactiver":                  COLORS["danger"],
#     "Dormants":                     COLORS["neutral"],
# }
#
# fig, axes = plt.subplots(1, 3, figsize=(18, 6))
#
# ── Donut : répartition des segments
# dist_plot  = dist.set_index("segment")
# plot_colors = [colors_seg.get(s, COLORS["neutral"]) for s in dist_plot.index]
# axes[0].pie(
#     dist_plot["nb_clients"], labels=dist_plot.index, colors=plot_colors,
#     autopct="%1.1f%%", wedgeprops=dict(edgecolor="white", linewidth=2),
#     startangle=90, pctdistance=0.75, labeldistance=1.12, textprops={"fontsize": 9}
# )
# axes[0].add_patch(plt.Circle((0, 0), 0.55, color="white"))
# axes[0].set_title("Répartition des segments RFM", fontweight="bold")
#
# ── Bar : CA par segment
# ca_seg = (
#     rfm.merge(df_liv.groupby("customer_id")["revenue"].sum().reset_index(), on="customer_id", how="left")
#     .groupby("segment_rfm")["revenue"].sum().sort_values(ascending=False)
# )
# seg_colors = [colors_seg.get(s, COLORS["neutral"]) for s in ca_seg.index]
# axes[1].bar(ca_seg.index, ca_seg.values / 1000, color=seg_colors, alpha=0.85, edgecolor="white")
# axes[1].set_title("CA total par segment (k€)", fontweight="bold")
# axes[1].tick_params(axis="x", rotation=25)
#
# ── Scatter : Recency vs Monetary
# for seg, grp in rfm.groupby("segment_rfm"):
#     axes[2].scatter(grp["recency"], grp["monetary"] / 1000,
#                     alpha=0.55, s=25, label=seg, color=colors_seg.get(seg, COLORS["neutral"]))
# axes[2].set_xlabel("Recency (jours)")
# axes[2].set_ylabel("CA client (k€)")
# axes[2].set_title("Recency vs Monetary par segment", fontweight="bold")
# axes[2].legend(fontsize=8, markerscale=1.5)
#
# plt.suptitle("Segmentation RFM — ShopAfrica+", fontsize=13, fontweight="bold", y=1.01)
# plt.tight_layout()
# plt.show()

---
## Étape 6 — Statistiques par segment

In [ ]:
# TODO : Calcule les stats moyennes par segment
# stats_seg = (
#     rfm.groupby("segment_rfm")
#     .agg(
#         nb_clients    = ("customer_id",  "count"),
#         recency_moy   = ("recency",      # TODO),
#         frequency_moy = ("frequency",    # TODO),
#         monetary_moy  = ("monetary",     # TODO),
#         r_moy         = ("R_score",      # TODO),
#         f_moy         = ("F_score",      # TODO),
#         m_moy         = ("M_score",      # TODO)
#     )
#     .round(1)
#     .sort_values("monetary_moy", ascending=False)
#     .reset_index()
# )
# display(stats_seg)

---
## Étape 7 — Interprétation business

Complète le tableau ci-dessous à partir de tes résultats :

| Segment | Nb clients | CA moyen/client | Action recommandée |
|---|---|---|---|
| Champions | ___ | ___ | Programme VIP, early access, referral |
| Fidèles | ___ | ___ | Upsell, cross-sell, offres premium |
| Gros dépensiers occasionnels | ___ | ___ | Incentive fréquence, abonnement |
| Nouveaux prometteurs | ___ | ___ | Welcome journey, nurturing |
| À réactiver | ___ | ___ | Email de réactivation, offre de retour |
| Dormants | ___ | ___ | CRM allégé, désabonnement |

> **📌 Point pédagogique** :
> La segmentation RFM n'est pas une fin en soi. Sa valeur tient dans les **actions différenciées** qu'elle permet.  
> Un Champion ne doit pas recevoir le même email qu'un Dormant.

---
## Étape 8 — Export du dataset RFM

Le dataset RFM segmenté sera utilisé dans le **Notebook 4 (Power BI)**.

In [ ]:
# Export CSV (récupérable depuis Colab via Files → Download)
# rfm.to_csv("clients_rfm_segments.csv", index=False)
# print("✅ clients_rfm_segments.csv exporté")
# print(f"   Colonnes : {rfm.columns.tolist()}")
# print(f"   Lignes   : {len(rfm):,} clients segmentés")

---
## Checklist de validation

**Partie SQL — DuckDB :**
- [ ] Bloc A — KPIs globaux
- [ ] Bloc B — Analyse temporelle (mensuel, top mois, anomalies avec LAG)
- [ ] Bloc C — Catégories (ranking avec RANK OVER)
- [ ] Bloc D — Clients (top 20, HAVING)
- [ ] Bloc E — Canaux (part CA avec window function)
- [ ] Bloc F — Risque (CASE WHEN)
- [ ] Bloc G — SQL avancé (ROW_NUMBER PARTITION BY, LAG, CTEs multiples)

**Partie Python — RFM :**
- [ ] Métriques RFM calculées
- [ ] Scores R/F/M via quantiles
- [ ] Segments définis avec les règles métier
- [ ] Visualisation produite
- [ ] Statistiques par segment calculées
- [ ] Interprétation business complétée
- [ ] Export `clients_rfm_segments.csv`

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.